In [4]:
# Day 31: Feature Selection and Data Optimization (Correlation, SelectKBest, Removing Useless Features)
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=3, n_redundant=7, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# RFE with Random Forest
selector = RFE(estimator=RandomForestClassifier(), n_features_to_select=3)
selector.fit(X_train, y_train)

X_train_reduced = selector.transform(X_train)
X_test_reduced = selector.transform(X_test)

# Simple Model (Bina XGBoost ke)
rf_final = RandomForestClassifier()
rf_final.fit(X_train_reduced, y_train)

preds = rf_final.predict(X_test_reduced)
print(f"Accuracy with RF: {accuracy_score(y_test, preds):.2f}")

Accuracy with RF: 0.93


In [5]:
# Day 31: Feature Selection and Data Optimization (Correlation, SelectKBest, Removing Useless Features)
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Data Creation: Artificial Dataset generate kar rahe hain
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5, n_redundant=5, random_state=42)
feature_names = [f"Feature_{i}" for i in range(10)]
df = pd.DataFrame(X, columns=feature_names)

# 2. Correlation Analysis: Highly correlated features ko hatana
# Yeh step redundant features ko remove karta hai
corr_matrix = df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.90)]
df = df.drop(columns=to_drop)
print(f"Dropped correlated columns: {to_drop}")

# 3. Feature Selection: Sirf best features chunna
# f_classif (ANOVA) use karke top 5 features nikal rahe hain
selector = SelectKBest(score_func=f_classif, k=5)
X_selected = selector.fit_transform(df, y)
selected_indices = selector.get_support(indices=True)
selected_features = [df.columns[i] for i in selected_indices]
print(f"Top 5 Selected Features: {selected_features}")

# 4. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

# 5. Model Building
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 6. Evaluation
predictions = model.predict(X_test)
print("\n--- Model Performance ---")
print(f"Accuracy: {accuracy_score(y_test, predictions):.2f}")
print("\nClassification Report:\n", classification_report(y_test, predictions))

Dropped correlated columns: []
Top 5 Selected Features: ['Feature_2', 'Feature_3', 'Feature_5', 'Feature_6', 'Feature_7']

--- Model Performance ---
Accuracy: 0.94

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.95      0.94       100
           1       0.95      0.93      0.94       100

    accuracy                           0.94       200
   macro avg       0.94      0.94      0.94       200
weighted avg       0.94      0.94      0.94       200

